# TFM — ¿Personalizar la recomendación de método supera a una elección fija?

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cecimerelo/ml-sandbox/blob/main/notebooks/tfm_graficos.ipynb)

Este cuaderno reproduce los **gráficos y el contraste estadístico** del Trabajo Fin de Máster a
partir de los resultados del estudio benchmark. Sigue la estructura exigida:

1. **Origen de los datos** — qué datasets se usaron y por qué son una muestra amplia y diversa.
2. **Técnica** — benchmark de 20 métodos de ISLR con validación cruzada (10 o 5 particiones), y
   evaluación *leave-one-dataset-out* de cuatro estrategias de selección de método.
3. **Resultados y contraste de hipótesis** — ¿acierta más la recomendación personalizada?
4. **Robustez** — valores faltantes y coste de exigir interpretabilidad.

> **Objetivo del TFM: el recomendador *Layer 2*.** Un modelo entrenado sobre los resultados del
> benchmark que, a partir de las características del problema del usuario, recomienda qué
> método de aprendizaje supervisado usar. En los gráficos aparece como **Aprendida (Layer 2)**,
> siempre en azul. Las demás estrategias son los puntos de comparación contra los que se mide.

**Hipótesis principal:** la recomendación personalizada (basada en las características del problema)
elige el mejor método con más frecuencia que una recomendación fija.
**Hipótesis secundaria:** las heurísticas didácticas de ISLR aciertan mejor que el azar.

Todo el código del estudio está en [github.com/cecimerelo/ml-sandbox](https://github.com/cecimerelo/ml-sandbox).
Basta con ejecutar las celdas en orden (*Entorno de ejecución → Ejecutar todas*).

## 0. Preparación

En Colab todas las dependencias vienen instaladas. En local hay dos opciones: instalarlas con
Poetry desde `notebooks/` (`poetry install`) y elegir `notebooks/.venv` como kernel, o usar
cualquier entorno de Python 3.12+: la primera celda instala con `pip` lo que falte.

Se clona el repositorio (público) para disponer de los resultados. El benchmark completo
—38.820 evaluaciones— tardó horas, así que el cuaderno parte de sus resultados guardados:

- `data/results/results-801f29e62585.parquet` — una fila por dataset × método × % de faltantes × partición.
- `data/results/outcomes.parquet` — qué eligió cada estrategia en cada dataset (evaluación *leave-one-dataset-out*).
- `data/results/method_scores.parquet` — puntuación media de cada método en cada dataset y si cuenta entre los mejores.
- `data/results/explainability.json` — coste de exigir interpretabilidad.

Los tres últimos se generan con `scripts/export_outcomes.py`; la sección final explica cómo recalcularlos.

**¿Importa que estos tres ficheros se hayan calculado después del benchmark?** No. No son una
reconstrucción aproximada de lo que ocurrió durante el estudio, sino el mismo cálculo repetido:

- **Son deterministas.** Se obtienen a partir del parquet de resultados con el código del
  repositorio y una semilla fija (20260815). Con los mismos resultados, dan siempre exactamente
  las mismas cifras, se calculen el día del benchmark o meses después.
- **Así se calculó siempre.** El estudio nunca guardó "lo que eligió cada estrategia en su
  momento": para cada dataset, las estrategias se reconstruyen a partir de los resultados de los
  otros 105 (*leave-one-dataset-out*) y se mira qué eligen. Las cifras de los informes del
  estudio se obtuvieron con ese mismo procedimiento; estos ficheros solo guardan además el
  detalle por dataset, en vez de únicamente la media.
- **Está comprobado.** Repetir el cálculo sobre la copia de los resultados con la que se
  redactaron las conclusiones originales reproduce exactamente sus cifras (por ejemplo, 0,47 de
  hit rate para la estrategia aprendida y para la fija).

**¿Por qué entonces algunas cifras difieren un poco de las conclusiones originales?** Por un
cambio en los datos, no en el cálculo. Ridge elige su parámetro de regularización (`alpha`)
probando varios valores por validación cruzada. Al principio solo probaba los tres que trae
scikit-learn por defecto (0,1, 1 y 10); más tarde se amplió a 25 valores entre 0,001 y 1.000 y
se volvió a ejecutar **solo Ridge**. Con más opciones, Ridge mejoró en 10 de los 106 datasets.

Como "acertar" significa elegir uno de los mejores métodos de cada dataset, que Ridge mejore
cambia en algunos datasets qué métodos cuentan como mejores, y eso mueve ligeramente los
aciertos de las estrategias (por ejemplo, la aprendida pasa de 0,47 a 0,49). Las conclusiones no
cambian. Este cuaderno usa los resultados corregidos.

Comprueba que están instaladas las librerías que usa el cuaderno (instala con
`pip` las que falten) y las importa todas.

In [1]:
import importlib.util
import json
import subprocess
import sys
from pathlib import Path

# Instala lo que falte en el entorno actual, para poder usar el cuaderno sin Poetry.
# módulo que se importa → paquete que se instala
# nbformat y jinja2 no se usan directamente: Plotly necesita nbformat para mostrar los gráficos
# en el cuaderno, y pandas necesita jinja2 para dar formato a las tablas (`.style`).
DEPENDENCIAS = {"numpy": "numpy", "pandas": "pandas", "pyarrow": "pyarrow",
                "plotly": "plotly", "scipy": "scipy", "nbformat": "nbformat>=4.2.0",
                "jinja2": "jinja2>=3.1"}
faltan = [paquete for modulo, paquete in DEPENDENCIAS.items()
          if importlib.util.find_spec(modulo) is None]
if faltan:
    print("Instalando:", ", ".join(faltan))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *faltan], check=True)
    # Sin esto, Python puede no ver los paquetes recién instalados en esta misma sesión.
    importlib.invalidate_caches()

import nbformat  # noqa: F401  (lo requiere fig.show() de Plotly)
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

Instalando: jinja2>=3.1


Los datos del estudio viven en el repositorio. Si el cuaderno se
ejecuta dentro de una copia local se usa esa, si no (en Colab), se clona desde GitHub.

In [2]:
REPO = "https://github.com/cecimerelo/ml-sandbox.git"

def encontrar_repo() -> Path:
    # En local el cuaderno vive en notebooks/; en Colab se clona el repositorio.
    for candidato in (Path.cwd(), Path.cwd().parent):
        if (candidato / "config" / "collection.json").exists():
            return candidato
    destino = Path("/content/ml-sandbox")
    if not destino.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO, str(destino)], check=True)
    return destino

ROOT = encontrar_repo()
print("Repositorio en", ROOT)

Repositorio en /Users/cecimerelo/Desktop/personal/TFM


Ahora cargamos los resultados del benchmark, lo que eligió cada estrategia, las
puntuaciones por método y la descripción de la colección (datasets, meta-características y
métodos). La salida confirma el tamaño del estudio.

In [3]:
RESULTADOS = pd.read_parquet(ROOT / "data/results/results-801f29e62585.parquet")
OUTCOMES = pd.read_parquet(ROOT / "data/results/outcomes.parquet")
METODOS_SCORES = pd.read_parquet(ROOT / "data/results/method_scores.parquet")
EXPLICABILIDAD = json.loads((ROOT / "data/results/explainability.json").read_text())
COLECCION = json.loads((ROOT / "config/collection.json").read_text())
METAFEATURES = pd.DataFrame(json.loads((ROOT / "config/metafeatures.json").read_text())["datasets"])
METODOS = pd.DataFrame(json.loads((ROOT / "config/methods.json").read_text())["methods"])

print(f"{len(RESULTADOS):,} evaluaciones · {RESULTADOS.dataset.nunique()} datasets · "
      f"{RESULTADOS.method.nunique()} métodos · faltantes: "
      f"{sorted(RESULTADOS.missing_rate.unique())}")

38,820 evaluaciones · 106 datasets · 20 métodos · faltantes: [np.float64(0.0), np.float64(0.05), np.float64(0.25)]


### Las estrategias de selección de método

Cada estrategia recibe las características de un dataset y devuelve **una ordenación de los
métodos candidatos**, del más al menos recomendado. Todas se construyen igual: para evaluar un
dataset solo pueden aprender de los otros 105 (*leave-one-dataset-out*), de modo que ninguna ha
"visto" el problema sobre el que se la juzga. En los datos aparecen con su nombre en el código,
entre paréntesis.

El método de *leave-one-dataset-out*?** se ha usado porque reproduce la situación real del recomendador: un
usuario llega con *su* dataset, que el modelo nunca ha visto, y la pregunta del TFM es si en ese
caso acierta. Por eso, para cada uno de los 106 datasets, la estrategia se entrena con los otros
105 y se le pide una recomendación para el que se ha dejado fuera.

- **Se separan datasets enteros, no filas al azar.** El modelo aprende de una tabla con una fila
  por dataset y método. Si se separara al azar, parte de los métodos de un mismo dataset caería
  en entrenamiento y parte en prueba: el modelo "recordaría" ese dataset en lugar de predecirlo,
  y daría una cifra mejor de la real sin que nada fallara.
- **Se aplica a todas las estrategias**, también a las que no aprenden, así ninguna parte con
  ventaja. Este diseño destapó un error real: la estrategia fija elegía su método mirando
  también el dataset con el que se la iba a evaluar. Se corrigió; de no hacerlo, habría sido
  artificialmente difícil de batir.
- **Se deja fuera uno cada vez**, y no un grupo, porque con solo 106 datasets así cada modelo se
  entrena con el máximo posible (105), el más parecido al que usa la aplicación (entrenado con
  los 106); el resultado no depende de cómo se agrupen los datasets, y el coste (106
  entrenamientos de un modelo pequeño) es de pocos minutos.

La primera es **el modelo desarrollado y entrenado en este TFM**,las otras cuatro existen para
poder juzgarlo: dos cotas de referencia (fija y aleatoria), la alternativa que ofrece el libro
de texto (heurísticas) y la variante que usa la aplicación (híbrida).

- 🎯 **Aprendida — Layer 2: el recomendador del TFM** (`learned (Layer 2)`). La recomendación
  **personalizada**, y el objeto de estudio del trabajo. Un *random
  forest* entrenado sobre los resultados del benchmark aprende a predecir cuánto rendimiento
  perderá cada método frente al mejor, a partir de las siete meta-características del dataset:
  tipo de tarea, banda de filas, banda de predictores, régimen de dimensionalidad (observaciones
  por predictor), tipos de variable, nivel de valores faltantes y balance de clases. Recomienda
  los métodos de menor pérdida prevista.

  Se ha usado un *random forest* porque lo que el modelo tiene que aprender depende de
  combinaciones de características: tener pocas filas no afecta igual a una red neuronal, que
  necesita muchos datos, que a un árbol de decisión. Un modelo lineal solo suma efectos por
  separado, así que habría que indicarle a mano cada una de esas combinaciones; un bosque de
  árboles las encuentra por sí mismo, ya que cada rama divide primero por una característica y
  después por otra. El bosque es deliberadamente pequeño (300 árboles y al menos 3
  observaciones por hoja) porque la tabla de entrenamiento también lo es, unas 1.500 filas, una
  por cada combinación de dataset y método. Con un bosque más profundo, el modelo memorizaría
  los datasets concretos en lugar de aprender el patrón común a todos ellos.

  El modelo predice cuánto pierde cada método frente al mejor, y no directamente cuál es el
  ganador, por dos motivos. Con la regla de empate muchos datasets tienen varios ganadores, y un
  clasificador aprendería a considerar incorrecto un método que en realidad empata con el mejor.
  Además, la pérdida prevista ordena directamente las alternativas que muestra la aplicación.
  Medirla frente al mejor método de cada dataset, en lugar de usar la puntuación bruta, evita
  que el modelo dedique su capacidad a aprender qué datasets son fáciles, algo que no sirve para
  recomendar.

  La elección del random forest responde a este razonamiento, pero no se comparó con otros
  modelos que también podrían servir para esta tarea, como *gradient boosting* o una regresión
  lineal con interacciones. Es posible que alguno de ellos rinda mejor, y comprobarlo queda como
  línea de trabajo futuro.
- **Método único fijo** (`single best method`). No mira el problema: recomienda **siempre el
  mismo orden de métodos**, el de menor pérdida media frente al mejor en los datasets de
  entrenamiento. Es la comparación clave de la hipótesis: si personalizar no la supera, no
  merece la pena inspeccionar el problema del usuario. En este estudio ese método resulta ser
  Random Forest en todos los casos.
- **Heurísticas — ISLR** (`heuristics (ISLR)`). Las reglas de decisión didácticas de *An
  Introduction to Statistical Learning* (sesgo-varianza, supuestos de cada método, tamaño de
  muestra frente a número de predictores…) codificadas como reglas fijas. No aprende nada de
  los datos: es lo que "diría el libro".
- **Aleatoria** (`random choice`). Ordena los métodos al azar, con semilla fija. Es la cota
  inferior: lo que se obtiene sin mirar nada.
- **Híbrida** (`hybrid`). La que usa la aplicación: las restricciones del usuario (por
  ejemplo, exigir un modelo interpretable) descartan métodos, y la estrategia aprendida ordena
  los que quedan. Sin restricciones es idéntica a la aprendida, por eso se omite de los
  gráficos; solo interviene en el coste de la interpretabilidad (Gráfico 10).

**Preparación común.** Fija el color y la etiqueta en español de cada estrategia (la misma en
todos los gráficos), define la función `estilo` que da formato a las figuras y construye la
tabla `OUT`: una fila por dataset y estrategia, con si acertó y cuánto perdió. `OUT0` es esa
misma tabla sin valores faltantes, la que usan los resultados principales. La híbrida se
descarta aquí tras comprobar que coincide con la aprendida.

In [4]:
# Estilo común de los gráficos
FUENTE = "system-ui, -apple-system, 'Segoe UI', sans-serif"
TINTA, TINTA_2, TINTA_3 = "#0b0b0b", "#52514e", "#898781"
REJILLA, EJE, FONDO = "#e1e0d9", "#c3c2b7", "#fcfcfb"
AZUL, NARANJA, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
AZULES = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]

ESTRATEGIAS = {  # nombre en el código → (etiqueta, color)
    "learned (Layer 2)": ("Aprendida (Layer 2)", AZUL),
    "single best method": ("Método único fijo", NARANJA),
    "heuristics (ISLR)": ("Heurísticas (ISLR)", AQUA),
    "random choice": ("Aleatoria", TINTA_3),
}
ORDEN_ESTRATEGIAS = [etiqueta for etiqueta, _ in ESTRATEGIAS.values()]
COLOR_ESTRATEGIA = {etiqueta: color for etiqueta, color in ESTRATEGIAS.values()}
ETIQUETA_METODO = dict(zip(METODOS.name, METODOS.label))
BANDAS_FILAS = ["<500", "500-10k", ">10k"]

hibrida = OUTCOMES[OUTCOMES.strategy == "hybrid"].set_index(["missing_rate", "dataset"]).choice
aprendida = OUTCOMES[OUTCOMES.strategy == "learned (Layer 2)"].set_index(["missing_rate", "dataset"]).choice
# Sin restricciones, la híbrida elige exactamente lo mismo que la aprendida; por eso se omite
# de los gráficos. Si algún cambio en el código rompiera esa igualdad, esta línea lo avisaría.
assert (hibrida == aprendida).all(), "La híbrida ya no coincide con la aprendida: revisar"

OUT = OUTCOMES[OUTCOMES.strategy.isin(ESTRATEGIAS)].copy()
OUT["estrategia"] = OUT.strategy.map(lambda s: ESTRATEGIAS[s][0])
OUT = OUT.merge(METAFEATURES[["dataset", "rows", "task"]], on="dataset", how="left")
OUT0 = OUT[OUT.missing_rate == 0.0]


MARGEN_SUP = 150


def estilo(fig, titulo, subtitulo=None, alto=460):
    texto = f"<b>{titulo}</b>"
    if subtitulo:
        texto += f"<br><span style='font-size:13px;color:{TINTA_2}'>{subtitulo}</span>"
    fig.update_layout(
        template="simple_white",
        font=dict(family=FUENTE, size=13, color=TINTA),
        title=dict(text=texto, x=0.01, xanchor="left", y=0.97, yanchor="top"),
        paper_bgcolor=FONDO, plot_bgcolor=FONDO, height=alto,
        margin=dict(t=MARGEN_SUP, l=70, r=30, b=60),
        # La leyenda va entre el subtítulo y el área de dibujo, por encima de los títulos
        # de los subgráficos: se desplaza en píxeles, convertidos a la escala del papel.
        legend=dict(orientation="h", x=0, y=1 + 42 / (alto - MARGEN_SUP - 60),
                    yanchor="bottom", title=None, font=dict(color=TINTA_2)),
        hoverlabel=dict(font=dict(family=FUENTE, size=13), bgcolor="white"),
    )
    fig.update_xaxes(linecolor=EJE, tickcolor=EJE, tickfont_color=TINTA_2,
                     title_font_color=TINTA_2, showgrid=False)
    fig.update_yaxes(linecolor=EJE, tickcolor=EJE, tickfont_color=TINTA_2,
                     title_font_color=TINTA_2, showgrid=True, gridcolor=REJILLA)
    fig.update_annotations(font=dict(color=TINTA_2))
    return fig

## 1. Origen de los datos

La idea era conseguir datasets que cubrieran cada uno de los casos que el usuario podría llegar a elegir a través del form. 

La colección combina tres fuentes públicas: **OpenML-CC18** (clasificación, ≥ 500 filas),
**OpenML-CTR23** (regresión, ≥ 500 filas) y **PMLB** (ambas tareas, < 500 filas). De 267
candidatos, 195 cumplían los criterios de elegibilidad y se conservaron **106**, estratificados
por tamaño y tipo de tarea para que ningún régimen domine la comparación.

**Gráfico 1 — ¿Es la colección variada?** Para que la comparación generalice, los datasets
deben cubrir tamaños y dimensiones muy distintos. Cada punto es un dataset, situado por su
número de filas y de predictores (ambos en escala logarítmica); el color indica la fuente y la
forma el tipo de tarea.

In [5]:
datasets = pd.DataFrame(COLECCION["datasets"])
FUENTES = {"openml-cc18": "OpenML-CC18", "openml-ctr23": "OpenML-CTR23", "pmlb": "PMLB"}
TAREAS = {"regression": "Regresión", "binary classification": "Clasificación binaria",
          "multiclass classification": "Clasificación multiclase"}
datasets["fuente"] = datasets.source.map(FUENTES)
datasets = datasets.merge(METAFEATURES[["dataset", "task", "rows"]].rename(
    columns={"dataset": "name", "task": "tarea_mf", "rows": "banda_filas"}), on="name")
datasets["tarea"] = datasets.tarea_mf.map(TAREAS)

fig = px.scatter(
    datasets, x="rows", y="predictors", color="fuente", symbol="tarea",
    log_x=True, log_y=True, hover_name="name",
    hover_data={"rows": ":,", "predictors": True, "fuente": True, "tarea": True},
    color_discrete_sequence=[AZUL, NARANJA, AQUA],
    symbol_sequence=["circle", "diamond", "square"],
    labels={"rows": "Filas (escala log)", "predictors": "Predictores (escala log)",
            "fuente": "Fuente", "tarea": "Tarea"},
)
fig.update_traces(marker=dict(size=10, opacity=0.85, line=dict(width=1.5, color=FONDO)))
estilo(fig, f"Gráfico 1 · Los {len(datasets)} datasets van de {datasets.rows.min():,} a "
            f"{datasets.rows.max():,} filas y de {datasets.predictors.min()} a "
            f"{datasets.predictors.max()} predictores".replace(",", "."),
       "Cada punto es un dataset. Color: fuente · forma: tipo de tarea", alto=540).show()

**Gráfico 2 y tabla — ¿Está equilibrada?** Cuántos datasets hay de cada tamaño y tipo de
tarea, y de qué fuente sale cada uno. Comprueba que ningún régimen domina la muestra.

In [6]:
recuento = (datasets.groupby(["banda_filas", "tarea"]).size().rename("n").reset_index())
fig = px.bar(
    recuento, x="banda_filas", y="n", color="tarea", text="n",
    category_orders={"banda_filas": BANDAS_FILAS, "tarea": list(TAREAS.values())},
    color_discrete_sequence=[AZUL, NARANJA, AQUA],
    labels={"banda_filas": "Filas del dataset", "n": "Datasets", "tarea": "Tarea"},
)
fig.update_traces(textposition="inside", textfont_color="white",
                  marker_line=dict(width=2, color=FONDO))
estilo(fig, "Gráfico 2 · Composición de la colección",
       f"Datasets por tamaño y tipo de tarea · total {len(datasets)}").show()

pd.crosstab(datasets.fuente, datasets.tarea, margins=True, margins_name="Total")

tarea,Clasificación binaria,Clasificación multiclase,Regresión,Total
fuente,,,,
OpenML-CC18,18,13,0,31
OpenML-CTR23,0,0,35,35
PMLB,10,10,20,40
Total,28,23,55,106


## 2. Técnica: el benchmark

Cada uno de los 20 métodos del temario de ISLR se entrenó y evaluó con **validación cruzada** en
cada dataset aplicable (*balanced accuracy* en clasificación, R² en regresión). El número de
particiones depende de la fuente:

- **10 particiones** en los 66 datasets de OpenML (CC18 y CTR23): se usan las particiones
  predefinidas que publica OpenML, para que los resultados sean directamente comparables con
  los trabajos publicados sobre esas colecciones.
- **5 particiones** en los 40 datasets de PMLB, que no traen particiones propias: se generan
  estratificadas (en clasificación), barajadas y con semilla fija. Se eligieron 5 porque era el
  valor especificado por defecto en la configuración del estudio; coincide con uno de los dos
  valores que recomienda ISLR (k = 5 o k = 10) y, con datasets de 50 a 500 filas, deja al menos
  10 observaciones en cada partición de prueba.

Que unos datasets usen 10 particiones y otros 5 no sesga la comparación: todas las métricas se
calculan **dentro de cada dataset** antes de agregarse, y nunca se compara la puntuación de un
dataset con la de otro. Dos métodos se consideran **empatados** cuando su puntuación media está a menos de
una desviación estándar (la del mejor método) de la del mejor: en un dataset puede haber
varios "mejores métodos".

Ese ranking real es la referencia contra la que se mide cualquier estrategia de selección.

**Gráfico 3 — ¿Qué método gana más veces?** Cuenta en cuántos datasets cada método es el mejor
o empata con él. Distingue las victorias en solitario (azul oscuro) de los empates (azul
claro). Es lo que explica la fuerza de la estrategia fija: si un método queda casi siempre
entre los mejores, recomendarlo siempre ya acierta mucho.

In [7]:
ms0 = METODOS_SCORES[METODOS_SCORES.missing_rate == 0.0].copy()
ganadores_por_dataset = ms0.groupby("dataset").winner.sum()
ms0["tipo_victoria"] = np.where(
    ms0.dataset.map(ganadores_por_dataset) == 1, "Único mejor", "Empatado entre los mejores")
victorias = (ms0[ms0.winner].groupby(["method", "tipo_victoria"]).size()
             .unstack(fill_value=0).reindex(columns=["Único mejor", "Empatado entre los mejores"],
                                            fill_value=0))
aplicable = ms0.groupby("method").dataset.nunique()
victorias["total"] = victorias.sum(axis=1)
victorias = victorias.reindex(aplicable.index, fill_value=0).sort_values("total")
etiquetas = [ETIQUETA_METODO.get(m, m) for m in victorias.index]

fig = go.Figure()
for columna, color in (("Único mejor", AZULES[5]), ("Empatado entre los mejores", AZULES[2])):
    fig.add_bar(
        y=etiquetas, x=victorias[columna], name=columna, orientation="h",
        marker=dict(color=color, line=dict(width=2, color=FONDO)),
        customdata=np.stack([victorias.total, aplicable.reindex(victorias.index)], axis=1),
        hovertemplate="<b>%{y}</b><br>" + columna + ": %{x}<br>"
                      "Mejor (total): %{customdata[0]} de %{customdata[1]} datasets en los que se ejecutó<extra></extra>",
    )
fig.update_layout(barmode="stack")
fig.update_xaxes(title="Datasets en los que es (uno de) los mejores", showgrid=True,
                 gridcolor=REJILLA)
fig.update_yaxes(showgrid=False)
estilo(fig, f"Gráfico 3 · {ETIQUETA_METODO[victorias.total.idxmax()]} es el método que más veces resulta mejor",
       "Datasets en los que cada método es el mejor o empata con él (regla de 1 desviación estándar)",
       alto=640).show()

mas_ganador = victorias.total.idxmax()
print(f"Método más frecuentemente mejor: {ETIQUETA_METODO[mas_ganador]} "
      f"({victorias.total.max()} de {aplicable[mas_ganador]} datasets en los que se ejecutó)")

Método más frecuentemente mejor: Random Forest (66 de 106 datasets en los que se ejecutó)


**Gráfico 4 — ¿Cuánto pierde cada método en cada dataset?** Un mapa de calor con los métodos en
filas y los datasets en columnas. Cada celda muestra cuánto rendimiento pierde ese método
frente al mejor de ese dataset: claro significa cerca del mejor, oscuro lejos, y en blanco el
método no aplica o no pudo ejecutarse. Permite ver de un vistazo que ningún método gana
siempre, pero que unos pocos casi nunca quedan lejos.

In [8]:
# Pérdida de cada método frente al mejor de su dataset. 0 = es el mejor; en blanco = el método
# no aplica a la tarea o no pudo ejecutarse. Recortado en 0.25 para que la escala sea legible.
TOPE = 0.25
matriz = ms0.pivot(index="method", columns="dataset", values="regret").clip(upper=TOPE)
orden_metodos = matriz.mean(axis=1).sort_values().index
tarea_ds = METAFEATURES.set_index("dataset").task
orden_datasets = sorted(matriz.columns, key=lambda d: (tarea_ds[d] != "regression",
                                                       ganadores_por_dataset[d]))
matriz = matriz.loc[orden_metodos, orden_datasets]

fig = go.Figure(go.Heatmap(
    z=matriz.values, x=matriz.columns, y=[ETIQUETA_METODO.get(m, m) for m in matriz.index],
    colorscale=[[i / (len(AZULES) - 1), c] for i, c in enumerate(AZULES)],
    zmin=0, zmax=TOPE, xgap=1, ygap=1,
    colorbar=dict(title=dict(text="Pérdida frente<br>al mejor", side="top"), thickness=12,
                  tickvals=[0, 0.1, 0.2, TOPE], ticktext=["0 (mejor)", "0.10", "0.20", "≥ 0.25"]),
    hovertemplate="<b>%{y}</b> en %{x}<br>Pérdida frente al mejor: %{z:.3f}<extra></extra>",
))
n_reg = int((tarea_ds[orden_datasets] == "regression").sum())
fig.add_vline(x=n_reg - 0.5, line=dict(color=TINTA_2, width=1.5, dash="dot"))
fig.add_annotation(x=n_reg / 2, y=1.0, yref="paper", yanchor="bottom", text="<b>Regresión</b>",
                   showarrow=False)
fig.add_annotation(x=n_reg + (len(orden_datasets) - n_reg) / 2, y=1.0, yref="paper",
                   yanchor="bottom", text="<b>Clasificación</b>", showarrow=False)
fig.update_xaxes(showticklabels=False, title="Datasets (ordenados por tarea y nº de métodos empatados)")
fig.update_yaxes(showgrid=False, autorange="reversed")
estilo(fig, "Gráfico 4 · Ningún método gana siempre, pero unos pocos casi nunca quedan lejos",
       "Pérdida de cada método frente al mejor de cada dataset · métodos ordenados de menor a mayor pérdida media",
       alto=680).show()

### Cómo se mide cada estrategia

Las estrategias (definidas en la sección 0) se evalúan sobre ese ranking real con tres métricas:

- **Hit rate:** la primera recomendación está entre los mejores métodos del dataset.
- **Top-3:** algún mejor método aparece entre las tres primeras sugerencias, que es lo que el
  usuario ve en la aplicación.
- **Regret:** rendimiento perdido por seguir la primera recomendación en lugar de usar el mejor
  método. Un método que no puede ejecutarse pierde la puntuación entera.

Se informan dos estratos: **todos los datasets (106)** y **solo aquellos en los que la elección
importa (51)**, es decir, donde empatan como mucho tres métodos. En los demás casi cualquier
elección acierta y la comparación dice poco.

## 3. Resultados y contraste de las hipótesis

**Tabla de resultados principal.** Hit rate, top-3 y regret medios de cada estrategia, en los
dos estratos. Es la tabla que responde a la hipótesis; los gráficos siguientes la desglosan.

In [9]:
def resumen(frame):
    return frame.groupby("estrategia").agg(
        datasets=("dataset", "nunique"), hit_rate=("hit", "mean"),
        top3=("top_k_hit", "mean"), regret=("regret", "mean")).reindex(ORDEN_ESTRATEGIAS)

tabla = pd.concat({"Todos los datasets": resumen(OUT0),
                   "Donde la elección importa": resumen(OUT0[OUT0.discriminating])})
tabla.round(3)

datasets  hit_rate   top3  \
                          estrategia                                       
Todos los datasets        Aprendida (Layer 2)       106     0.632  0.783   
                          Método único fijo         106     0.623  0.745   
                          Heurísticas (ISLR)        106     0.321  0.679   
                          Aleatoria                 106     0.264  0.594   
Donde la elección importa Aprendida (Layer 2)        51     0.490  0.686   
                          Método único fijo          51     0.471  0.647   
                          Heurísticas (ISLR)         51     0.176  0.490   
                          Aleatoria                  51     0.020  0.294   

                                               regret  
                          estrategia                   
Todos los datasets        Aprendida (Layer 2)   0.041  
                          Método único fijo     0.036  
                          Heurísticas (ISLR)    0.116  
                          Aleatoria             0.244  
Donde la elección importa Aprendida (Layer 2)   0.038  
                          Método único fijo     0.031  
                          Heurísticas (ISLR)    0.112  
                          Aleatoria             0.325

**Gráfico 5 — ¿Es real la diferencia entre estrategias?** Los mismos aciertos de la tabla, con
su **intervalo de confianza al 95 %** calculado por *bootstrap*: se remuestrean los datasets
2.000 veces y se mide cuánto varía el resultado. Si los intervalos de dos estrategias se
solapan mucho, la diferencia entre ellas puede deberse al azar de qué datasets entraron en la
muestra.

In [10]:
RNG = np.random.default_rng(20260815)
REPETICIONES = 2000

def intervalo(valores, rng=RNG, reps=REPETICIONES):
    # IC 95 % bootstrap sobre datasets: se remuestrean datasets, no evaluaciones.
    valores = np.asarray(valores, dtype=float)
    medias = rng.choice(valores, size=(reps, len(valores)), replace=True).mean(axis=1)
    return np.percentile(medias, [2.5, 97.5])

filas = []
for estrato, frame in (("Todos los datasets (106)", OUT0),
                       ("Donde la elección importa (51)", OUT0[OUT0.discriminating])):
    for metrica, columna in (("Hit rate", "hit"), ("Top-3", "top_k_hit")):
        for estrategia in ORDEN_ESTRATEGIAS:
            valores = frame.loc[frame.estrategia == estrategia, columna]
            bajo, alto = intervalo(valores)
            filas.append(dict(estrato=estrato, metrica=metrica, estrategia=estrategia,
                              valor=valores.mean(), bajo=bajo, alto=alto))
ic = pd.DataFrame(filas)

estratos = ic.estrato.unique().tolist()
metricas = ["Hit rate", "Top-3"]
fig = make_subplots(rows=2, cols=2, shared_yaxes=True, vertical_spacing=0.16,
                    horizontal_spacing=0.05,
                    subplot_titles=[f"{m} · {e}" for m in metricas for e in estratos])
for i, metrica in enumerate(metricas, start=1):
    for j, estrato in enumerate(estratos, start=1):
        sub = ic[(ic.metrica == metrica) & (ic.estrato == estrato)]
        for _, fila in sub.iterrows():
            fig.add_bar(
                x=[fila.estrategia], y=[fila.valor], name=fila.estrategia,
                marker=dict(color=COLOR_ESTRATEGIA[fila.estrategia]),
                error_y=dict(type="data", symmetric=False, array=[fila.alto - fila.valor],
                             arrayminus=[fila.valor - fila.bajo], color=TINTA_2, thickness=1.5),
                text=[f"{fila.valor:.2f}"], textposition="inside", insidetextanchor="start",
                textfont=dict(color="white"),
                showlegend=(i == 1 and j == 1), legendgroup=fila.estrategia,
                hovertemplate=f"<b>{fila.estrategia}</b><br>{metrica}: {fila.valor:.2f}<br>"
                              f"IC 95 %: [{fila.bajo:.2f}, {fila.alto:.2f}]<extra></extra>",
                row=i, col=j)
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(range=[0, 1.08], tickformat=".0%")
estilo(fig, "Gráfico 5 · La recomendación personalizada empata con la elección fija",
       "Proporción de aciertos por estrategia, con intervalo de confianza bootstrap al 95 %",
       alto=720).show()

**Contraste de la hipótesis principal.** Se comparan la estrategia aprendida y el método único
fijo **sobre los mismos datasets** (datos pareados):

- *Hit rate* y *top-3*: **test exacto de McNemar** — solo cuentan los datasets en los que una
  acierta y la otra no.
- *Regret*: **test de rangos con signo de Wilcoxon** sobre la diferencia de pérdida por dataset.

In [11]:
def pareado(frame, a="Aprendida (Layer 2)", b="Método único fijo"):
    ancho = frame.pivot(index="dataset", columns="estrategia",
                        values=["hit", "top_k_hit", "regret"])
    filas = []
    for metrica, columna in (("Hit rate", "hit"), ("Top-3", "top_k_hit")):
        acierta = ancho[columna].astype(bool)
        gana_a = int((acierta[a] & ~acierta[b]).sum())
        gana_b = int((~acierta[a] & acierta[b]).sum())
        p = stats.binomtest(gana_a, gana_a + gana_b).pvalue if gana_a + gana_b else 1.0
        filas.append(dict(métrica=metrica, **{f"gana {a}": gana_a, f"gana {b}": gana_b},
                          p_valor=p))
    diferencia = (ancho["regret"][a] - ancho["regret"][b]).astype(float)
    distintos = diferencia[diferencia != 0]
    p = stats.wilcoxon(distintos).pvalue if len(distintos) else 1.0
    filas.append(dict(métrica="Regret", **{f"gana {a}": int((diferencia < 0).sum()),
                                            f"gana {b}": int((diferencia > 0).sum())},
                      p_valor=p))
    return pd.DataFrame(filas).set_index("métrica")

contraste = pd.concat({"Todos los datasets": pareado(OUT0),
                       "Donde la elección importa": pareado(OUT0[OUT0.discriminating])})
contraste.style.format({"p_valor": "{:.3f}"})

**Gráfico 6 — ¿Cómo fallan?** La media del regret no distingue entre fallar un poco muchas
veces y fallar mucho unas pocas. Aquí cada punto es un dataset y la caja resume su
distribución: se ve si una estrategia comete errores graves (puntos altos) o solo pequeños.

In [12]:
fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.05,
                    subplot_titles=["Todos los datasets (106)", "Donde la elección importa (51)"])
for j, frame in enumerate((OUT0, OUT0[OUT0.discriminating]), start=1):
    for estrategia in ORDEN_ESTRATEGIAS:
        sub = frame[frame.estrategia == estrategia]
        fig.add_box(
            x=[estrategia] * len(sub), y=sub.regret, name=estrategia,
            marker=dict(color=COLOR_ESTRATEGIA[estrategia], size=7, opacity=0.6,
                        line=dict(width=1, color=FONDO)),
            line=dict(color=COLOR_ESTRATEGIA[estrategia], width=2),
            fillcolor="rgba(0,0,0,0)", boxpoints="all", jitter=0.45, pointpos=0,
            boxmean=True, customdata=sub.dataset,
            hovertemplate="<b>%{customdata}</b><br>Regret: %{y:.3f}<extra>" + estrategia + "</extra>",
            showlegend=(j == 1), legendgroup=estrategia, row=1, col=j)
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(title_text="Regret (rendimiento perdido)", row=1, col=1)
estilo(fig, "Gráfico 6 · Las heurísticas y el azar fallan a veces por mucho",
       "Regret por dataset · caja = cuartiles, línea discontinua = media, cada punto un dataset",
       alto=560).show()

**Gráfico 7 — ¿Ganan en los mismos datasets?** Compara, dataset a dataset, lo que pierden la
aprendida y la fija. Los puntos sobre la diagonal son empates; por debajo gana la aprendida,
por encima la fija. Un empate en la media podría esconder que cada una gana en datasets
distintos: este gráfico lo muestra.

In [13]:
ancho = OUT0.pivot(index="dataset", columns="estrategia", values="regret")
ancho = ancho.join(OUT0.drop_duplicates("dataset").set_index("dataset")[["discriminating"]])
a, b = ancho["Aprendida (Layer 2)"], ancho["Método único fijo"]
ancho["resultado"] = np.select([np.isclose(a, b), a < b, a > b],
                               ["Igual regret", "Aprendida pierde menos", "Fija pierde menos"],
                               default="Igual regret")
conteo = ancho.resultado.value_counts()
colores = {"Aprendida pierde menos": AZUL, "Fija pierde menos": NARANJA,
           "Igual regret": TINTA_3}
tope = float(max(a.max(), b.max())) * 1.05

fig = go.Figure()
fig.add_scatter(x=[0, tope], y=[0, tope], mode="lines", showlegend=False, hoverinfo="skip",
                line=dict(color=EJE, width=1.5, dash="dash"))
for resultado, color in colores.items():
    sub = ancho[ancho.resultado == resultado]
    fig.add_scatter(
        x=sub["Método único fijo"], y=sub["Aprendida (Layer 2)"], mode="markers",
        name=f"{resultado} ({conteo.get(resultado, 0)})",
        marker=dict(color=color, size=10, opacity=0.8, line=dict(width=1.5, color=FONDO)),
        customdata=sub.index,
        hovertemplate="<b>%{customdata}</b><br>Fija: %{x:.3f}<br>Aprendida: %{y:.3f}<extra></extra>")
fig.add_annotation(x=tope * 0.75, y=tope * 0.2, text="La aprendida pierde menos ↓",
                   showarrow=False, font=dict(color=TINTA_2))
fig.add_annotation(x=tope * 0.2, y=tope * 0.8, text="↑ La fija pierde menos",
                   showarrow=False, font=dict(color=TINTA_2))
fig.update_xaxes(title="Regret del método único fijo", range=[-0.01, tope], showgrid=True,
                 gridcolor=REJILLA, constrain="domain")
fig.update_yaxes(title="Regret de la estrategia aprendida", range=[-0.01, tope],
                 scaleanchor="x", scaleratio=1, constrain="domain")
estilo(fig, "Gráfico 7 · Dataset a dataset, ninguna de las dos domina",
       f"Cada punto es un dataset ({len(ancho)}). Sobre la diagonal, ambas pierden lo mismo",
       alto=680).show()

**Gráfico 8 — ¿Depende del tamaño del dataset?** Hit rate de cada estrategia según el número de
filas. Comprueba si la personalización ayuda en algún régimen concreto aunque no lo haga en
conjunto. La tabla final contrasta la banda de 500 a 10.000 filas, donde una versión anterior
del estudio parecía mostrar una ventaja.

In [14]:
por_banda = (OUT0.groupby(["rows", "estrategia"])
             .agg(hit_rate=("hit", "mean"), datasets=("dataset", "nunique")).reset_index())
por_banda_d = (OUT0[OUT0.discriminating].groupby(["rows", "estrategia"])
               .agg(hit_rate=("hit", "mean"), datasets=("dataset", "nunique")).reset_index())

fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.05,
                    subplot_titles=["Todos los datasets", "Donde la elección importa"])
for j, frame in enumerate((por_banda, por_banda_d), start=1):
    n = frame.groupby("rows").datasets.first().reindex(BANDAS_FILAS)
    for estrategia in ORDEN_ESTRATEGIAS:
        sub = frame[frame.estrategia == estrategia].set_index("rows").reindex(BANDAS_FILAS)
        fig.add_bar(
            x=[f"{banda}<br><span style='font-size:11px'>n = {n[banda]}</span>"
               for banda in BANDAS_FILAS],
            y=sub.hit_rate, name=estrategia, marker=dict(color=COLOR_ESTRATEGIA[estrategia]),
            marker_line=dict(width=2, color=FONDO), showlegend=(j == 1), legendgroup=estrategia,
            hovertemplate="<b>" + estrategia + "</b><br>Filas %{x}<br>Hit rate: %{y:.2f}<extra></extra>",
            row=1, col=j)
fig.update_layout(barmode="group", bargap=0.25)
fig.update_xaxes(title_text="Filas del dataset")
fig.update_yaxes(tickformat=".0%", range=[0, 1], title_text="Hit rate", row=1, col=1)
estilo(fig, "Gráfico 8 · Por tamaño de muestra, la aprendida tampoco supera a la fija",
       "Hit rate por banda de filas del dataset", alto=520).show()

(OUT0[OUT0.discriminating & (OUT0.rows == "500-10k")]
   .pipe(pareado).style.format({"p_valor": "{:.3f}"})
   .set_caption("Contraste pareado en la banda 500–10k, donde la elección importa"))

,gana Aprendida (Layer 2),gana Método único fijo,p_valor
métrica,,,
Hit rate,2,2,1.000
Top-3,2,0,0.500
Regret,5,3,0.641


## 4. Robustez

### 4.1 Valores faltantes

El benchmark se repitió eliminando al azar un **5 %** y un **25 %** de las celdas de los
predictores. ¿Cambia la conclusión cuando los datos están incompletos?

**Gráfico 9 — ¿Se mantiene el resultado con datos incompletos?** Hit rate y regret de cada
estrategia con 0 %, 5 % y 25 % de celdas faltantes, sobre los 106 datasets. La tabla recoge los
valores exactos.

In [15]:
por_tasa = (OUT.groupby(["missing_rate", "estrategia"])
            .agg(hit_rate=("hit", "mean"), regret=("regret", "mean")).reset_index())

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.1,
                    subplot_titles=["Hit rate (más es mejor)", "Regret (menos es mejor)"])
for j, metrica in enumerate(("hit_rate", "regret"), start=1):
    for estrategia in ORDEN_ESTRATEGIAS:
        sub = por_tasa[por_tasa.estrategia == estrategia]
        fig.add_scatter(
            x=sub.missing_rate, y=sub[metrica], name=estrategia, mode="lines+markers",
            line=dict(color=COLOR_ESTRATEGIA[estrategia], width=2),
            marker=dict(size=9, line=dict(width=1.5, color=FONDO)),
            showlegend=(j == 1), legendgroup=estrategia,
            hovertemplate="<b>" + estrategia + "</b><br>Faltantes: %{x:.0%}<br>"
                          + ("Hit rate: %{y:.2f}" if metrica == "hit_rate" else "Regret: %{y:.3f}")
                          + "<extra></extra>",
            row=1, col=j)
fig.update_xaxes(tickvals=[0, 0.05, 0.25], tickformat=".0%", title_text="Celdas faltantes")
fig.update_yaxes(tickformat=".0%", range=[0, 1], row=1, col=1)
fig.update_yaxes(rangemode="tozero", row=1, col=2)
estilo(fig, "Gráfico 9 · El empate se mantiene con datos incompletos",
       "Todos los datasets (106) con 0 %, 5 % y 25 % de celdas faltantes", alto=500).show()

(por_tasa.pivot(index="estrategia", columns="missing_rate", values=["hit_rate", "regret"])
   .reindex(ORDEN_ESTRATEGIAS).round(3))

hit_rate               regret              
missing_rate            0.00   0.05   0.25   0.00   0.05   0.25
estrategia                                                     
Aprendida (Layer 2)    0.632  0.726  0.689  0.041  0.038  0.035
Método único fijo      0.623  0.764  0.689  0.036  0.032  0.033
Heurísticas (ISLR)     0.321  0.302  0.292  0.116  0.125  0.132
Aleatoria              0.264  0.302  0.274  0.244  0.234  0.208

**Contraste por nivel de faltantes.** El mismo test pareado de la hipótesis principal
(aprendida frente a fija), repetido para cada porcentaje de celdas faltantes.

In [16]:
contrastes_tasa = pd.concat({
    f"{tasa:.0%} faltantes": pareado(OUT[OUT.missing_rate == tasa])
    for tasa in sorted(OUT.missing_rate.unique())})
contrastes_tasa.style.format({"p_valor": "{:.3f}"}).set_caption(
    "Aprendida frente a fija, por nivel de faltantes (todos los datasets)")

### 4.2 Coste de exigir interpretabilidad

El recomendador permite al usuario pedir un modelo explicable: con **"algo importante"** se
descartan los métodos opacos (bagging, random forest, boosting, SVM RBF, redes neuronales); con
**"crítica"** también los que solo se explican con esfuerzo (KNN, QDA, PCR/PLS, polinomios,
splines, GAM). ¿Cuánto rendimiento cuesta esa restricción?

**Gráfico 10 — ¿Cuánto cuesta la interpretabilidad?** Regret medio de la estrategia del
recomendador sin restricciones y con cada nivel de exigencia de interpretabilidad. La diferencia
entre barras es el rendimiento que se sacrifica a cambio de un modelo explicable.

In [17]:
niveles = {n["explainability"]: n for n in EXPLICABILIDAD["levels"]}
base = niveles["somewhat"]["unconstrained_mean_regret"]
barras = pd.DataFrame([
    ("Sin restricción", base, None),
    ("Algo importante", niveles["somewhat"]["mean_regret"],
     niveles["somewhat"]["datasets_where_it_bound"]),
    ("Crítica", niveles["critical"]["mean_regret"], niveles["critical"]["datasets_where_it_bound"]),
], columns=["nivel", "regret", "limita"])
total = RESULTADOS.dataset.nunique()

fig = go.Figure(go.Bar(
    x=barras.nivel, y=barras.regret, marker=dict(color=[AZULES[2], AZULES[4], AZULES[6]]),
    text=[f"{r:.3f}" for r in barras.regret], textposition="outside", cliponaxis=False,
    textfont=dict(color=TINTA),
    customdata=[("—" if pd.isna(l) else f"{int(l)} de {total}") for l in barras.limita],
    hovertemplate="<b>%{x}</b><br>Regret medio: %{y:.3f}<br>"
                  "Datasets en los que la restricción cambia la elección: %{customdata}<extra></extra>",
))
fig.update_xaxes(title="Importancia de la interpretabilidad")
fig.update_yaxes(title="Regret medio", rangemode="tozero")
veces = niveles["somewhat"]["mean_regret"] / base
estilo(fig, f"Gráfico 10 · Exigir interpretabilidad multiplica el regret por {veces:.1f}".replace(".", ","),
       f"Estrategia del recomendador · la restricción cambia la elección en "
       f"{niveles['somewhat']['datasets_where_it_bound']} de {total} datasets", alto=480).show()

## 5. Conclusiones

- **Hipótesis principal — no se verifica.** Personalizar la recomendación según las
  características del problema no acierta más que recomendar siempre el mismo método
  (Gráficos 5 y 7): los hit rates son prácticamente iguales, las diferencias no son
  significativas en ninguna métrica y el resultado se mantiene por tamaño de muestra
  (Gráfico 8) y con datos incompletos (Gráfico 9).
- **Hipótesis secundaria — se verifica, con matices.** Las heurísticas de ISLR aciertan
  claramente más que el azar, pero quedan muy por detrás de la elección fija y pierden
  bastante más rendimiento cuando fallan (Gráficos 5 y 6).
- **Hallazgo no anticipado.** El propio benchmark revela cuál es esa elección fija tan difícil
  de batir —Random Forest, con Gradient Boosting muy cerca (Gráfico 3)— y, con ello, que en datos tabulares típicos elegir
  bien el método importa **menos de lo que se suele suponer** (Gráfico 4).
- **Interpretabilidad.** Exigir un modelo explicable tiene un coste real y medible: cambia la
  elección en casi todos los datasets, porque el mejor método disponible suele ser opaco
  (Gráfico 10).

## Anexo — Recalcular todo desde los resultados

Los ficheros `outcomes.parquet`, `method_scores.parquet` y `explainability.json` se generan a
partir del parquet de resultados con el código del repositorio. Para recalcularlos (unos
15–20 minutos; requiere Python 3.12), cambia `RECALCULAR` a `True` y vuelve a ejecutar el
cuaderno desde el principio después de esta celda.

In [18]:
RECALCULAR = False

if RECALCULAR:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)], check=True)
    subprocess.run([sys.executable, "scripts/export_outcomes.py"], cwd=ROOT, check=True)